In [1]:
!pip3 install pandas numpy matplotlib seaborn scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip3 install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from graph import Graph,Path

In [3]:
parking_permit_blocks = pd.read_csv(r'../data/residential_parking_permit_blocks.csv')
street_nodes = pd.read_csv(r'../data/Street_Nodes.csv')
street_centerlines = pd.read_csv(r'../data/Street_Centerline.csv')
#N/S North Side
#S/S South Side
#E/S East Side
#W/S West Side
#B/S Both Sides

In [4]:
pd.set_option('display.max_columns', None)
parking_permit_blocks
street_nodes

,X,Y,objectid,arc_,streetcl_,streetcl_i,node_id,int_id,update_,intersecti,x_coord,y_coord
0,-8.367363e+06,4.860134e+06,1,NaN,NaN,NaN,29152,29152,2006/11/02 00:00:00+00,N 16TH ST & CALLOWHILL ST,NaN,NaN
1,-8.367186e+06,4.860106e+06,2,NaN,NaN,NaN,29150,29150,2006/11/02 00:00:00+00,N 15TH ST & CALLOWHILL ST,NaN,NaN
2,-8.367152e+06,4.860318e+06,3,NaN,NaN,NaN,29148,29148,2013/02/28 00:00:00+00,N 15TH ST & HAMILTON ST,NaN,NaN
3,-8.367538e+06,4.860163e+06,4,NaN,NaN,NaN,29025,29025,2006/05/03 00:00:00+00,N 17TH ST & CALLOWHILL ST,NaN,NaN
4,-8.367547e+06,4.860104e+06,5,NaN,NaN,NaN,29021,29021,2006/05/03 00:00:00+00,N 17TH ST & CARLTON ST,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
25596,-8.352965e+06,4.869284e+06,25597,NaN,NaN,NaN,30209,30209,2026/01/16 10:04:41+00,SB DELAWARE EXPY,NaN,NaN
25597,-8.364182e+06,4.874956e+06,25598,NaN,NaN,NaN,30210,30210,2026/01/16 11:16:01+00,OLD YORK RD,NaN,NaN
25598,-8.372243e+06,4.860246e+06,25599,NaN,NaN,NaN,30211,30211,2026/02/06 07:33:29+00,BUSTI ST & MANDELA WAY,NaN,NaN
25599,-8.372484e+06,4.860181e+06,25600,NaN,NaN,NaN,30212,30212,2026/02/06 08:03:29+00,N MELVILLE ST & SAINT MALACHYS WAY,NaN,NaN


In [5]:
pd.set_option('display.max_columns', None)
complete_streets = pd.read_csv(r'../data/CompleteStreets.csv')
complete_streets.TNODE.max()

np.float64(25133.0)

In [6]:
full_ds = pd.merge(street_centerlines,street_nodes[['X','Y','intersecti','node_id']],left_on=['fnode_'],right_on='node_id',validate='many_to_one',how='left',suffixes=('','_from')).rename({'X':'X_from','Y':'Y_from','intersecti':'intersecti_from'},axis=1)
full_ds  = pd.merge(full_ds,street_nodes[['X','Y','intersecti','node_id']],left_on=['tnode_'],right_on='node_id',validate='many_to_one',how='left',suffixes=('','_to')).rename({'X':'X_to','Y':'Y_to','intersecti':'intersecti_to'},axis=1)

In [7]:
dx = full_ds['X_to'] - full_ds['X_from']
dy = full_ds['Y_to'] - full_ds['Y_from']

full_ds['bearing'] = np.degrees(np.arctan2(dy,dx))

In [8]:
def get_valid_optins(df,column):
    print(df[column].value_counts())

In [9]:
full_ds[(full_ds.oneway.str.strip() != '')].head(1)

,fnode_,tnode_,lpoly_,rpoly_,length,stcl2_,stcl2_id,pre_dir,st_name,st_type,suf_dir,zip_left,zip_right,l_f_add,l_t_add,r_f_add,r_t_add,st_code,l_hundred,r_hundred,seg_id,oneway,class,responsibl,update_,newsegdate,multi_rep,streetlabe,stname,objectid,Shape__Length,X_from,Y_from,intersecti_from,node_id,X_to,Y_to,intersecti_to,node_id_to,bearing
0,26705,26716,NaN,NaN,2862.844615,NaN,NaN,,BARTRAM,AVE,,19153,19153,9200,9298,9201.0,9299.0,16120,9200,9200,100002,B,2,STATE,2004/05/18 00:00:00+00,NaN,0.0,BARTRAM AVE,BARTRAM AVE,1,1137.344551,-8.377195e+06,4.848714e+06,BARTRAM AVE & DELAWARE EXPY RAMP TP,26705,-8.378059e+06,4.848000e+06,BARTRAM AVE,26716,-140.45888


In [10]:
philly_streets = Graph()

In [11]:
for ind, data in full_ds[(full_ds.oneway.str.strip() != '')].iterrows():
    fnode = data.fnode_
    tnode = data.tnode_
    length = data.length
    stname = data.stname
    oneway = data.oneway
    from_intersection = data.intersecti_from
    to_intesection = data.intersecti_to
    bearing = data.bearing
    opposite = (bearing + 180) % 360
    if opposite > 180:
        opposite -= 360
    if oneway == 'B':
        philly_streets.add_edge(from_node_id=fnode,
                                to_node_id=tnode,
                                from_node_intersection = from_intersection,
                                to_node_intersection = to_intesection,
                                length=length,
                                stname = stname,
                                oneway=oneway,
                                bearing=bearing)
        philly_streets.add_edge(from_node_id=tnode,
                                to_node_id=fnode,
                                from_node_intersection = from_intersection,
                                to_node_intersection = to_intesection,
                                length=length,
                                stname = stname,
                                oneway=oneway,
                                bearing=opposite)
    elif oneway == 'TF':
        philly_streets.add_edge(from_node_id=tnode,
                                to_node_id=fnode,
                                from_node_intersection = from_intersection,
                                to_node_intersection = to_intesection,
                                length=length,
                                stname = stname,
                                oneway=oneway,
                                bearing=opposite)
    elif oneway == 'FT':
        philly_streets.add_edge(from_node_id=fnode,
                                to_node_id=tnode,
                                from_node_intersection = from_intersection,
                                to_node_intersection = to_intesection,
                                length=length,
                                stname = stname,
                                oneway=oneway,
                                bearing=bearing)
        

In [12]:
philly_streets.build_adjacency()

In [13]:
x = philly_streets.show_node_adjacency(3)

In [14]:
philly_streets.adjacency    

defaultdict(list,
            {<graph.Node at 0x1133286e0>: [<graph.Edge at 0x113328440>,
             <graph.Node at 0x112df3610>: [<graph.Edge at 0x112df3890>,
             <graph.Node at 0x112df3750>: [<graph.Edge at 0x112df39d0>,
             <graph.Node at 0x112e3d810>: [<graph.Edge at 0x112e3d940>,
             <graph.Node at 0x112e3dba0>: [<graph.Edge at 0x112ea1e10>,
             <graph.Node at 0x1133d4170>: [<graph.Edge at 0x112ea17b0>,
             <graph.Node at 0x112ea18c0>: [<graph.Edge at 0x112e90350>,
             <graph.Node at 0x112ea2030>: [<graph.Edge at 0x112e92c50>,
             <graph.Node at 0x1133d8250>: [<graph.Edge at 0x112e9bc50>,
             <graph.Node at 0x1133d8150>: [<graph.Edge at 0x112e9bd40>,
             <graph.Node at 0x11332da90>: [<graph.Edge at 0x112ee1b70>,
             <graph.Node at 0x11332d9a0>: [<graph.Edge at 0x112ee2190>],
             <graph.Node at 0x112ee20b0>: [<graph.Edge at 0x1133e9300>,
             <graph.Node at 0x112ee1d30>: [<g

In [15]:
path = Path(x.from_node_obj[0])
print(path)
path.drive_edge(x.edge[0])
print(path)

COUNTY LINE RD & OVERHILL AVE
COUNTY LINE RD & OVERHILL AVE -> OVERHILL AVE -> BUXMONT ST & OVERHILL AVE


In [20]:
x

""
